# Phase 1 — Data Loading and Validation

This notebook inspects the untouched `cardekho_dataset.csv`. It does **not** clean, impute, remove, cap, or transform any values. The goal is to create a reproducible record of the raw dataset before Phase 2 begins.

## 1. Imports and project paths

The path logic works when the notebook kernel starts either in the project root or inside the `notebooks` folder.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cardekho_dataset.csv"
SUMMARY_PATH = PROJECT_ROOT / "reports" / "metrics" / "data_validation_summary.json"

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dataset: {DATA_PATH}")

## 2. Load the raw dataset

`pd.read_csv()` reads the file into a DataFrame. We keep the raw columns exactly as supplied so the validation can detect the exported index and other issues.

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

## 3. Schema, data types, and memory usage

In [ ]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
    "unique_values": df.nunique(dropna=False).values,
})

memory_mb = df.memory_usage(index=True, deep=True).sum() / 1024**2
print(f"Deep memory usage: {memory_mb:.2f} MB")
display(schema)

## 4. Missing values and duplicates

Duplicate listings must be measured after excluding `Unnamed: 0`, because that exported index gives otherwise identical rows different values. No rows are removed in this notebook.

In [ ]:
missing_summary = (
    df.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_percent=lambda table: table["missing_count"] / len(df) * 100)
)

full_row_duplicates = int(df.duplicated().sum())
listing_duplicates = int(
    df.drop(columns=["Unnamed: 0"], errors="ignore").duplicated().sum()
)

display(missing_summary)
print(f"Full-row duplicates with exported index: {full_row_duplicates}")
print(f"Duplicate listings without exported index: {listing_duplicates}")

## 5. Numerical ranges and target distribution

The 95th and 99th percentiles help distinguish the common range from extreme values. They are inspection thresholds here, not automatic deletion rules.

In [ ]:
numeric_columns = [
    "vehicle_age", "km_driven", "mileage", "engine",
    "max_power", "seats", "selling_price",
]

numeric_profile = (
    df[numeric_columns]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .T
)

display(numeric_profile)
print(f"Selling-price skewness: {df['selling_price'].skew():.2f}")
print(f"Median selling price: ₹{df['selling_price'].median():,.0f}")
print(f"Mean selling price: ₹{df['selling_price'].mean():,.0f}")

## 6. Categorical coverage

Small categories may need low-confidence warnings in the final recommendation application.

In [ ]:
categorical_columns = [
    "car_name", "brand", "model", "seller_type",
    "fuel_type", "transmission_type",
]

for column in categorical_columns:
    counts = df[column].value_counts(dropna=False).head(15)
    print(f"\n{column}: {df[column].nunique(dropna=False)} unique values")
    display(counts.rename("row_count").to_frame())

## 7. Integrity checks

These checks identify suspicious or redundant values without changing them.

In [ ]:
expected_car_name = (
    df["brand"].astype(str).str.strip()
    + " "
    + df["model"].astype(str).str.strip()
)

checks = pd.Series({
    "negative_numeric_values": int((df[numeric_columns] < 0).sum().sum()),
    "zero_seat_rows": int(df["seats"].eq(0).sum()),
    "km_above_500000_rows": int(df["km_driven"].gt(500_000).sum()),
    "nonpositive_target_rows": int(df["selling_price"].le(0).sum()),
    "car_name_matches_brand_model_rows": int(df["car_name"].eq(expected_car_name).sum()),
})

display(checks.rename("count").to_frame())

In [ ]:
extreme_km_rows = (
    df.loc[df["km_driven"].gt(500_000), [
        "car_name", "vehicle_age", "km_driven", "selling_price"
    ]]
    .sort_values("km_driven", ascending=False)
)

zero_seat_rows = df.loc[df["seats"].eq(0)]

print("Rows above 500,000 km:")
display(extreme_km_rows)
print("Rows with zero seats:")
display(zero_seat_rows)

## 8. Save the reproducible validation summary

The reusable module records the schema, file hash, missing values, duplicates, percentiles, category coverage, integrity checks, and warnings in JSON. It still performs no cleaning.

In [ ]:
from src.validate_data import save_summary, summarize_dataframe

validation_summary = summarize_dataframe(df, DATA_PATH)
save_summary(validation_summary, SUMMARY_PATH)

print(f"Validation summary saved to: {SUMMARY_PATH}")
print(f"Warnings recorded: {len(validation_summary['warnings'])}")
for warning in validation_summary["warnings"]:
    print(f"- {warning}")

## Phase 1 conclusion

The raw file is structurally complete but not yet modeling-ready. Phase 2 must remove the exported index and duplicates, resolve zero-seat records without leakage, drop redundant `car_name` in the primary table, and investigate extreme odometer readings. Luxury prices and rare fuel categories must not be discarded automatically.